# CSRats — exploração dos jogadores

Usa os mesmos módulos dos scripts (`hurst_csrats`, `indicadores_csrats`, `caracteristicas_csrats`),
então os números batem com o painel e com os CSVs de resultado.

- **Kernel:** escolha o Python de `.venv` (canto superior direito no VS Code).
- **Atualizar dados:** troque o CSV da planilha e rode tudo de novo (*Run All*).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from caracteristicas_csrats import DIMENSOES, caracterizar, confiabilidade
from hurst_csrats import CSV_PADRAO, ler_jogadores

AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "lines.linewidth": 2})
pd.set_option("display.max_columns", 50)

## 1. Calcular tudo
Leva ~10 s por causa dos testes de embaralhamento.

In [ ]:
jogadores = ler_jogadores(CSV_PADRAO)
ctx = caracterizar(jogadores)
por_nick = {j["nick"]: j for j in jogadores}
DIMS = [nome for nome, _ in DIMENSOES]
TESTES = ["tendencia", "autocorr", "sequencias", "cusum"]

df = pd.DataFrame([{
    "nick": j["nick"], "nome": j["nome"], "n": len(j["partidas"]),
    "conf": confiabilidade(len(j["partidas"])),
    **{k: j["ind"].get(k) for k in ["media", "media_encolhida", "dp", "cv", "piso_p10", "teto_p90",
                                      "assimetria", "forma_10", "forma_z", "tendencia_10", "autocorr_lag1"]},
    "hurst8": j["hurst8"] and j["hurst8"]["h_corrigido"],
    **{f"pct_{d}": j["pct"][d] for d in DIMS},
    **{f"{t}_q": j["ind"].get(f"{t}_q") for t in TESTES},
    "fortes": ", ".join(j["fortes"]), "fracos": ", ".join(j["fracos"]), "perfil": ", ".join(j["perfil"]),
} for j in jogadores]).set_index("nick")

ref = df[df.n >= 20]  # referência dos percentis
print(f"{len(df)} jogadores, {len(ref)} na referência; média da liga {ctx['media_liga']:.1f}")

## 2. Tabela de características (percentis)
`*` = indicativo: só vale como característica real se o teste tiver q < 0,05 (seção 5).

In [ ]:
cols = [f"pct_{d}" for d in DIMS]
(ref.sort_values("media_encolhida", ascending=False)[["n", "media"] + cols + ["perfil"]]
    .rename(columns={f"pct_{d}": d + ("*" if d in ("Momento", "Evolução", "Fases") else "") for d in DIMS})
    .style.format(precision=0, na_rep="—").format({"media": "{:.1f}"})
    .background_gradient(cmap="Blues", vmin=0, vmax=100, subset=[d + ("*" if d in ("Momento", "Evolução", "Fases") else "") for d in DIMS]))

## 3. Quais dimensões são realmente diferentes?
Correlação entre as dimensões nos jogadores da referência. Valores perto de ±1 = medem quase a mesma coisa.

In [ ]:
corr = ref[cols].corr().rename(index=lambda c: c[4:], columns=lambda c: c[4:])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(DIMS)), corr.columns, rotation=45, ha="right"); ax.set_yticks(range(len(DIMS)), corr.index)
for i in range(len(DIMS)):
    for k in range(len(DIMS)):
        ax.text(k, i, f"{corr.iloc[i, k]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, k]) > .6 else "black")
ax.grid(False); fig.colorbar(im, shrink=.8); ax.set_title("Correlação entre dimensões")
plt.show()

## 4. Mapa da liga: nível × consistência
As duas dimensões que mais separam os jogadores. Tamanho do ponto = número de partidas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.scatter(ref.media_encolhida, ref.cv, s=ref.n * 3, color=AZUL, alpha=.6, edgecolor="white", linewidth=1.5)
ax.invert_yaxis()  # mais consistente em cima
fig.canvas.draw()
render, ocupados = fig.canvas.get_renderer(), []  # caixas dos rótulos já colocados
opcoes = [(6, 3, "left"), (6, -10, "left"), (-6, 3, "right"), (-6, -10, "right"),
          (0, 10, "center"), (0, -17, "center"), (8, 14, "left"), (-8, -21, "right")]
for nick, r in ref.sort_values("n", ascending=False).iterrows():
    for dx, dy, ha in opcoes:
        rot = ax.annotate(nick, (r.media_encolhida, r.cv), xytext=(dx, dy), textcoords="offset points", fontsize=8, ha=ha)
        caixa = rot.get_window_extent(render).expanded(1.05, 1.15)
        if not any(caixa.overlaps(o) for o in ocupados):
            break
        rot.remove()
    else:  # nenhuma posição livre: usa a primeira mesmo
        rot = ax.annotate(nick, (r.media_encolhida, r.cv), xytext=opcoes[0][:2], textcoords="offset points", fontsize=8)
        caixa = rot.get_window_extent(render)
    ocupados.append(caixa)
ax.axvline(ctx["media_liga"], color=CINZA, ls="--", lw=1); ax.axhline(ref.cv.median(), color=CINZA, ls="--", lw=1)
ax.set_xlabel("Nível (média encolhida de ADR)"); ax.set_ylabel("Coeficiente de variação (menor = mais consistente)")
ax.set_title("Bons e estáveis no canto superior direito")
plt.show()

## 5. Efeitos confirmados
Testes que resistem ao ajuste para vários jogadores (q < 0,05).

In [ ]:
qcols = [f"{t}_q" for t in TESTES]
confirmados = df[qcols].lt(.05).any(axis=1)
df.loc[confirmados, ["n", "tendencia_10", "autocorr_lag1"] + qcols].style.format(precision=3, na_rep="—").highlight_between(left=0, right=.05, subset=qcols, color="#cde2fb")

## 6. Ficha de um jogador
Troque os nicks na última linha. Use `ficha("A", "B")` para comparar.

In [ ]:
def radar(ax, j, cor, tracejado=False):
    ang = np.linspace(0, 2 * np.pi, len(DIMS), endpoint=False)
    v = [j["pct"][d] if j["pct"][d] is not None else 0 for d in DIMS]
    ax.plot(np.r_[ang, ang[0]], v + v[:1], color=cor, ls="--" if tracejado else "-", label=j["nick"])
    ax.fill(np.r_[ang, ang[0]], v + v[:1], color=cor, alpha=.12)
    ax.set_xticks(ang, DIMS); ax.set_ylim(0, 100); ax.spines["polar"].set_alpha(.2); ax.set_yticks([25, 50, 75], [])


def ficha(a, b=None):
    ja, jb = por_nick[a], por_nick.get(b)
    fig = plt.figure(figsize=(13, 4.5))
    ax1 = fig.add_subplot(1, 3, 1, projection="polar")
    ax2 = fig.add_subplot(1, 3, (2, 3))
    for j, cor, tr in [(jb, LARANJA, True), (ja, AZUL, False)]:
        if not j:
            continue
        radar(ax1, j, cor, tr)
        x = np.arange(1, len(j["partidas"]) + 1)
        ax2.plot(x, j["partidas"], color=cor, alpha=.3, lw=1, marker="o", ms=3)
        ax2.plot(x, pd.Series(j["partidas"]).rolling(min(10, len(x))).mean(), color=cor, label=f"{j['nick']} (média das últ. 10)")
    ax2.axhline(ctx["media_liga"], color=CINZA, ls="--", lw=1, label="média da liga")
    ax2.set_xlabel("partida (ordem cronológica)"); ax2.set_ylabel("ADR"); ax2.legend(frameon=False)
    if jb:
        ax1.legend(loc="lower right", bbox_to_anchor=(1.25, -.1), frameon=False)
    fig.suptitle(" × ".join(j["nick"] for j in (ja, jb) if j))
    plt.tight_layout(); plt.show()
    cols_f = ["n", "conf", "media", "media_encolhida", "cv", "piso_p10", "teto_p90", "forma_10", "tendencia_10", "fortes", "fracos", "perfil"]
    return df.loc[[j["nick"] for j in (ja, jb) if j], cols_f]


ficha("vnx", "Shaylong")

## 7. Espaço livre
`df` tem uma linha por jogador; `por_nick["vnx"]["partidas"]` é a série de ADR em ordem cronológica.

In [ ]:
df.describe().T